# Kiểm chứng split PKU và chặn leakage
**Mục tiêu:** đọc manifest + danh sách ảnh của phase 09; kiểm tra trùng group augmentation và near-duplicate từ **mọi ảnh** bằng `imagehash.phash` (Hamming ≤ 5), kể cả cross-split. **Input:** `data/splits/{train,val,test}.txt`, `split_manifest.json`, `data/processed/{images,labels}/`. **Output:** bảng kiểm tra và `docs/leakage_report.md`. Nếu FAIL, notebook vẫn hiển thị kết quả và kết thúc bình thường; **dừng pipeline tại đây**: phase 11 chỉ chấp nhận báo cáo PASS cùng digest manifest. Thống kê size: diện tích chuẩn hóa về 640×640, small < 32², medium 32²–96² (gồm hai biên), large > 96². Không train, không chạm DeepPCB.

In [1]:
from pathlib import Path
from collections import defaultdict
from IPython.display import display, Markdown
from PIL import Image
import hashlib, json, math, re
import pandas as pd
try: import imagehash
except ImportError as exc: raise ImportError('Cài ImageHash: python -m pip install ImageHash rồi chạy lại phase 10') from exc
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/splits/split_manifest.json').is_file()), None)
if ROOT is None: raise FileNotFoundError('Thiếu manifest; chạy 08 rồi 09 trước')
manifest_path = ROOT / 'data/splits/split_manifest.json'
manifest_bytes = manifest_path.read_bytes()
manifest = json.loads(manifest_bytes)
if manifest['dataset'] != 'PKU-Market-PCB(Data enhanced version)': raise ValueError('Manifest không phải PKU')
items = manifest['items']
splits = ('train', 'val', 'test')
if not items: raise ValueError('Manifest rỗng')
classes = re.findall(r"['\"]([^'\"]+)['\"]", (ROOT / 'data/raw/PKU-Market-PCB(Data enhanced version)/Class.txt').read_text(encoding='utf-8-sig'))
expected = ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper']
if classes != expected: raise ValueError('Class.txt PKU sai thứ tự: ' + str(classes))


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
errors, group_splits = [], defaultdict(set)
image_splits = defaultdict(set)
for row in items:
    if row['split'] not in splits:
        errors.append(f'Split không hợp lệ: {row}')
        continue
    group = re.sub(r'_[123]$', '', Path(row['source_image']).stem)
    if row['group_id'] != group: errors.append(f'Group ID sai: {row["source_image"]}')
    group_splits[group].add(row['split'])
    image_splits[row['image']].add(row['split'])
    image, label, source = (ROOT / row[key] for key in ('image', 'label', 'source_image'))
    if not image.is_symlink() or not source.is_file() or image.resolve() != source.resolve(): errors.append(f'Ảnh processed không trỏ đến PKU gốc: {image}')
    if not label.is_file() or label != ROOT / 'data/processed/labels' / row['source_split'] / (image.stem + '.txt'): errors.append(f'Label processed không khớp: {image}')
for split in splits:
    path = ROOT / 'data/splits' / (split + '.txt')
    if not path.is_file():
        errors.append(f'Thiếu {path}')
        continue
    content = path.read_bytes()
    if hashlib.sha256(content).hexdigest() != manifest['lists_sha256'][split]: errors.append(f'Digest {split}.txt khác manifest')
    listed = content.decode('utf-8').splitlines()
    expected_list = [str((ROOT / r['image']).absolute()) for r in items if r['split'] == split]
    if listed != expected_list: errors.append(f'{split}.txt khác thứ tự/nội dung manifest')
group_leaks = {g: sorted(ss) for g, ss in group_splits.items() if len(ss) > 1}
errors.extend(f'Ảnh có trong nhiều split: {p}' for p, ss in image_splits.items() if len(ss) > 1)
if len(image_splits) != len(items): errors.append('Manifest chứa ảnh lặp')
display(pd.DataFrame([{'check': 'Group leakage', 'status': 'FAIL' if group_leaks else 'PASS', 'violating_groups': len(group_leaks)}, {'check': 'Manifest / danh sách / symlink', 'status': 'FAIL' if errors else 'PASS', 'errors': len(errors)}]))
display(Markdown('Bảng kiểm tra toàn vẹn: cùng stem bỏ hậu tố augmentation phải thuộc đúng một split; danh sách, label và symlink phải khớp manifest.'))


,check,status,violating_groups,errors
0,Group leakage,PASS,0.0,NaN
1,Manifest / danh sách / symlink,PASS,NaN,0.0


Bảng kiểm tra toàn vẹn: cùng stem bỏ hậu tố augmentation phải thuộc đúng một split; danh sách, label và symlink phải khớp manifest.

In [3]:
# Tính pHash thực tế; BK-tree tìm chính xác các cặp Hamming <= 5 (không dùng hash lưu từ audit cũ).
tree = None
cross_pairs, total_pairs = [], 0
hash_errors = []
parent = list(range(len(items)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def union(i, j):
    a, b = find(i), find(j)
    if a != b: parent[b] = a
first_by_group = {}
for i, row in enumerate(items):
    group = row['group_id']
    if group in first_by_group: union(i, first_by_group[group])
    else: first_by_group[group] = i
for i, row in enumerate(items):
    try:
        with Image.open(ROOT / row['image']) as im: value = int(str(imagehash.phash(im)), 16)
    except (OSError, ValueError) as exc:
        hash_errors.append(f'{row["image"]}: {exc}')
        continue
    if tree is None:
        tree = [value, [i], {}]
        continue
    stack = [tree]
    while stack:
        node = stack.pop()
        distance = (value ^ node[0]).bit_count()
        if distance <= 5:
            total_pairs += len(node[1])
            for j in node[1]:
                union(i, j)
                if items[i]['split'] != items[j]['split']: cross_pairs.append((j, i, distance))
        stack.extend(child for edge, child in node[2].items() if distance - 5 <= edge <= distance + 5)
    node = tree
    while True:
        distance = (value ^ node[0]).bit_count()
        if distance == 0:
            node[1].append(i)
            break
        child = node[2].get(distance)
        if child is None:
            node[2][distance] = [value, [i], {}]
            break
        node = child
errors.extend(hash_errors)
examples = pd.DataFrame([{'image_a': items[a]['image'], 'split_a': items[a]['split'], 'image_b': items[b]['image'], 'split_b': items[b]['split'], 'distance': d} for a,b,d in cross_pairs[:25]], columns=['image_a', 'split_a', 'image_b', 'split_b', 'distance'])
display(pd.DataFrame([{'check': 'Near-duplicate cross-split (pHash <= 5)', 'status': 'FAIL' if cross_pairs or hash_errors else 'PASS', 'pairs_cross_split': len(cross_pairs), 'pairs_all': total_pairs}]))
display(examples)
display(Markdown('Bảng gồm tối đa 25 cặp minh họa; số cặp phía trên tính trên **tất cả** ảnh PKU. pHash là phép sàng lọc ảnh gần giống, không phải chứng minh cùng một board.'))
components = defaultdict(list)
for i, row in enumerate(items): components[find(i)].append(row)
component_table = pd.DataFrame([{'images': len(members), 'augmentation_groups': len({r['group_id'] for r in members}), 'board_prefixes': ', '.join(sorted({r['group_id'].split('_')[0] for r in members})), 'splits': ', '.join(sorted({r['split'] for r in members}))} for members in components.values()]).sort_values('images', ascending=False).reset_index(drop=True)
if manifest.get('phash_near_pairs') != total_pairs or manifest.get('split_unit_count') != len(components):
    errors.append('Số cạnh/cụm pHash tính lại khác manifest; cần chạy phase 09 rồi phase 10.')
display(Markdown(f'### Cụm pHash + augmentation: {len(components)} cụm (mọi ảnh trong mỗi cụm phải cùng split để đạt pHash PASS)'))
display(component_table.head(25))
display(Markdown('Bảng này cho thấy mức độ có thể chia 70/15/15 nếu coi mọi cạnh pHash <=5 là cùng group. Cụm có nhiều tiền tố board phản ánh quan hệ bắc cầu, chưa chứng minh cùng board vật lý.'))


,check,status,pairs_cross_split,pairs_all
0,Near-duplicate cross-split (pHash <= 5),PASS,0,82495


,image_a,split_a,image_b,split_b,distance


Bảng gồm tối đa 25 cặp minh họa; số cặp phía trên tính trên **tất cả** ảnh PKU. pHash là phép sàng lọc ảnh gần giống, không phải chứng minh cùng một board.

### Cụm pHash + augmentation: 6 cụm (mọi ảnh trong mỗi cụm phải cùng split để đạt pHash PASS)

,images,augmentation_groups,board_prefixes,splits
0,2416,421,"01, 04, 05, 06, 07",train
1,240,60,08,val
2,240,60,09,val
3,240,60,11,test
4,240,60,12,test
5,128,32,10,val


Bảng này cho thấy mức độ có thể chia 70/15/15 nếu coi mọi cạnh pHash <=5 là cùng group. Cụm có nhiều tiền tố board phản ánh quan hệ bắc cầu, chưa chứng minh cùng board vật lý.

In [4]:
distribution = {s: dict.fromkeys(classes, 0) for s in splits}
sizes = {s: {'small': 0, 'medium': 0, 'large': 0} for s in splits}
for row in items:
    if row['split'] not in splits: continue
    image, label = ROOT / row['image'], ROOT / row['label']
    if not image.is_file() or not label.is_file(): continue
    try:
        with Image.open(image) as im: width, height = im.size
        lines = label.read_text(encoding='utf-8').splitlines()
        for n, line in enumerate(lines, 1):
            if not line.strip(): continue
            parts = line.split()
            if len(parts) != 5: raise ValueError(f'{label}:{n}: số cột khác 5')
            cid, cx, cy, w, h = map(float, parts)
            if not all(map(math.isfinite, (cid, cx, cy, w, h))) or not cid.is_integer() or not 0 <= cid < 6 or not all(0 <= x <= 1 for x in (cx, cy, w, h)) or w <= 0 or h <= 0 or cx-w/2 < -1e-9 or cy-h/2 < -1e-9 or cx+w/2 > 1+1e-9 or cy+h/2 > 1+1e-9: raise ValueError(f'{label}:{n}: YOLO box không hợp lệ')
            distribution[row['split']][classes[int(cid)]] += 1
            area_640 = w * width * h * height / (width * height) * 640**2
            sizes[row['split']]['small' if area_640 < 32**2 else 'medium' if area_640 <= 96**2 else 'large'] += 1
    except (OSError, ValueError) as exc: errors.append(str(exc))
class_table = pd.DataFrame(distribution).reindex(classes)
size_table = pd.DataFrame(sizes).reindex(['small','medium','large'])
size_percent = size_table.div(size_table.sum(axis=0).replace(0, float('nan')), axis=1).mul(100).round(2)
display(Markdown('## Số box của sáu lớp theo split'))
display(class_table)
display(Markdown('Mỗi ô là số box từ label processed theo class ID trong Class.txt, không phải số ảnh.'))
display(Markdown('## Size bins COCO quy về 640×640: số box và tỷ lệ (%) theo split'))
display(size_table)
display(size_percent)
display(Markdown('Tỷ lệ chia cho tổng box mỗi split; area_640 = w_norm × h_norm × 640²; đây không phải kích thước pixel của ảnh gốc.'))


## Số box của sáu lớp theo split

,train,val,test
Missing_hole,1518,508,408
Mouse_bite,1506,504,400
Open_circuit,1410,524,404
Short,1496,496,404
Spur,1482,500,404
Spurious_copper,1530,528,404


Mỗi ô là số box từ label processed theo class ID trong Class.txt, không phải số ảnh.

## Size bins COCO quy về 640×640: số box và tỷ lệ (%) theo split

,train,val,test
small,8226,2769,2235
medium,714,291,189
large,2,0,0


,train,val,test
small,91.99,90.49,92.2
medium,7.98,9.51,7.8
large,0.02,0.00,0.0


Tỷ lệ chia cho tổng box mỗi split; area_640 = w_norm × h_norm × 640²; đây không phải kích thước pixel của ảnh gốc.

In [5]:
status = 'FAIL' if (group_leaks or cross_pairs or errors) else 'PASS'
manifest_digest = hashlib.sha256(manifest_bytes).hexdigest()
report = '\n'.join([
    '# PKU split leakage report', '', f'**Overall: {status}**', f'Manifest SHA256: `{manifest_digest}`', '',
    f'Group leakage: {len(group_leaks)} groups; pHash Hamming <= 5 cross-split: {len(cross_pairs)} pairs (all near pairs: {total_pairs}); integrity errors: {len(errors)}.',
    'Quy tắc: group bỏ `_1`/`_2`/`_3` cuối stem; kiểm tra pHash trên mọi ảnh PKU. Nếu FAIL, dừng trước khi tạo data.yaml/train.', '',
    'pHash là hash toàn ảnh: PCB có layout giống nhau nhưng lỗi khác nhau có thể có khoảng cách nhỏ. PASS ở đây chứng minh không có cạnh pHash <= 5 khác split theo quy tắc đã chọn, không chứng minh mỗi ảnh có board vật lý khác nhau.', '',
    '## Connected split units (augmentation + pHash)', '```text', component_table.to_string(index=False), '```', '',
    '## Group leakage examples', '```text', str(list(group_leaks.items())[:20]), '```', '',
    '## Near-duplicate cross-split examples', '```text', examples.to_string(index=False), '```', '',
    '## Integrity issues (first 30)', '```text', '\n'.join(errors[:30]) or '(none)', '```', '',
    '## Boxes by class and split', '```text', class_table.to_string(), '```', '',
    '## COCO size bins @640: counts', '```text', size_table.to_string(), '```', '',
    '## COCO size bins @640: percent', '```text', size_percent.to_string(), '```', ''
]) + '\n'
target = ROOT / 'docs/leakage_report.md'
target.write_text(report, encoding='utf-8')
display(Markdown(f'**{status}** — đã ghi `{target.relative_to(ROOT)}`; báo cáo chứa digest manifest để phase 11 không dùng kết quả cũ.'))
if status == 'FAIL':
    display(Markdown('### ⛔ DỪNG PIPELINE — không chạy notebook 11, không dùng split để train.'))
    display(Markdown(f'Phát hiện **{len(group_leaks)} group bị tách**, **{len(cross_pairs)} cặp pHash gần trùng giữa các split**, **{len(errors)} lỗi toàn vẹn**. Xem cặp ví dụ ở trên và `docs/leakage_report.md` để điều tra; sau khi sửa split cần chạy lại phase 09–10. pHash của các PCB chung layout có thể giống nhau dù lỗi khác nhau: không tự coi mọi cặp là bản sao thật.'))
else:
    display(Markdown('**Đạt phase 10:** không phát hiện group hoặc pHash near-duplicate cross-split theo ngưỡng đã định; có thể chạy phase 11.'))


**PASS** — đã ghi `docs/leakage_report.md`; báo cáo chứa digest manifest để phase 11 không dùng kết quả cũ.

**Đạt phase 10:** không phát hiện group hoặc pHash near-duplicate cross-split theo ngưỡng đã định; có thể chạy phase 11.

## Nhận xét / điều kiện kết thúc
PASS chỉ khi group không bị tách, không có cặp pHash Hamming ≤ 5 khác split, và manifest/label/ảnh đều hợp lệ. FAIL được ghi và hiển thị mà không ném exception; **không tiếp tục phase 11**, vì phase 11 kiểm tra `Overall: PASS` và digest manifest. pHash có thể có false positive trên board giống nhau; cần điều tra các cặp báo FAIL trước khi quyết định sửa cách chia, không tự ý đổi ngưỡng để ép PASS.